# Finansielle Strategier, Tekniske Indikatorer & Feature Engineering

## Importering

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

import plotly.express as px

In [2]:
# Loading keys
load_dotenv()
KEY = os.getenv("key")
SECRET = os.getenv("secret")

# Loading data
client = StockHistoricalDataClient(KEY, SECRET)
df = client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols = "AAPL",
    timeframe = TimeFrame.Day,
    start = "2025-01-01"
)).df

df.describe()

,open,high,low,close,volume,trade_count,vwap
count,365.000000,365.000000,365.000000,365.000000,3.650000e+02,3.650000e+02,365.000000
mean,244.666304,247.453103,242.106013,244.814384,5.216435e+07,6.713360e+05,244.812574
std,30.695596,30.667242,30.654531,30.633163,2.116690e+07,2.303931e+05,30.639918
min,171.950000,190.335000,169.210100,172.420000,1.791057e+07,3.440390e+05,179.164809
25%,215.000000,218.760000,213.530000,215.240000,3.995995e+07,5.298490e+05,215.609629
50%,248.020000,249.690000,245.130000,247.450000,4.674254e+07,6.201120e+05,247.548096
75%,268.815000,271.875000,266.250000,269.050000,5.567230e+07,7.530750e+05,269.350428
max,314.175000,317.400000,309.650000,315.200000,1.843868e+08,2.212850e+06,313.143117


## Tekniske Indikatorer

### Simple Moving Average (SMA)

Et glidende gjennomsnitt som beregner gjennomsnittsprisen over en bestemt periode. SMA brukes ofte for å identifisere langsiktige trender:

$$
SMA=
\frac{1}{n}
\sum_{i=1}^{n}P_i
$$

F.eks:  
Kjøp når sma50 > sma100,  
Selg når sma50 < sma100


In [3]:
df["sma5"] = df["close"].rolling(5).mean()
df["sma50"] = df["close"].rolling(50).mean()
df["sma100"] = df["close"].rolling(100).mean()

px.line(
    df.dropna().reset_index(),
    x = "timestamp",
    y = ["sma5","sma50","sma100"]
).show()

### Exponential Moving Average (EMA)

Et glidende gjennomsnitt som gir nyere observasjoner høyere vekt. EMA reagerer raskere på nye prisbevegelser enn SMA:
$$
EMA_t=
\alpha P_t
+
(1-\alpha)EMA_{t-1}
$$

F.eks:  
Kjøp når close > ema20,  
Selg når close < ema20

In [4]:
df["ema20"] = df["close"].ewm(span = 20, adjust= False).mean()

px.line(
    df.dropna().reset_index(),
    x = "timestamp",
    y= ["ema20", "close"]
)

### Relative Strength Index (RSI)

Relative Strength er forholdet mellom gjennomsnittlig gevinst og gjennomsnittlig tap over en gitt periode. RS brukes som grunnlaget for beregning av Relative Strength Index (RSI):

$$
RS=
\frac{
Average\ Gain
}
{
Average\ Loss
}
$$

<br>

En momentumindikator som måler styrken i nylige prisbevegelser. RSI brukes ofte til å identifisere overkjøpte og oversolgte markeder:

$$
RSI=
100-
\frac{100}{1+RS}
$$

<br>

F.eks:  
Kjøp når RSI < 30,  
Selg når RSI > 70

In [5]:
# prisendringer 
delta = df["close"].diff()

# gain and loss
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

# avg gain / loss
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["rsi14"] = 100 - (100 / (1 + rs))

px.line(df.reset_index().dropna(), "timestamp", "rsi14")

### MACD

En momentum- og trendindikator basert på forskjellen mellom to eksponentielle glidende gjennomsnitt. MACD brukes ofte for å identifisere trendendringer:

$$
MACD=
EMA_{12}
-
EMA_{26}
$$

F.eks:  
Kjøp når > 0,  
Selg når < 0

In [6]:
ema12 = df["close"].ewm(span=12, adjust=False).mean()
ema26 = df["close"].ewm(span=26, adjust=False).mean()

macd = ema12-ema26

signal = macd.ewm(span = 9, adjust=False).mean()

df["macd"] = macd - signal

px.line(df.dropna().reset_index(), "timestamp", "macd")

### Bollinger Bands

En volatilitetsindikator som består av et glidende gjennomsnitt og to standardavviksbånd. Bollinger Bands brukes ofte til å identifisere perioder med høy eller lav volatilitet:

$$
Upper\ Band=
SMA+2\sigma
$$

$$
Lower\ Band=
SMA-2\sigma
$$

F.eks:  
Kjøp når Close < Lower_Band,  
Selg når Close > Upper_Band

In [7]:
middle = df["close"].rolling(20).mean()
std = df["close"].rolling(20).std()

upper = middle + 2 * std
lower = middle - 2 * std

df["upperBB"] = upper
df["lowerBB"] = lower

px.line(df.dropna().reset_index(), "timestamp", ["upperBB", "lowerBB", "close"])

### Average True Range (ATR)

En indikator som måler markedsvolatilitet. ATR brukes ofte til Risikohåndtering, Stop Loss-plassering og Position Sizing:

$$
ATR=
Moving\ Average(True\ Range)
$$

Stop_Loss = Entry - 2 * ATR,  
Entry = Nøyaktig tidspunktet og den kursen du kjøper deg inn i en aksje eller en posisjon på.

In [8]:
high_low = df["high"] - df["low"]
high_close = abs(df["high"] - df["close"].shift())
low_close = abs(df["low"] - df["close"].shift())

tr = pd.concat([high_low, high_close, low_close], axis = 1).max(axis = 1)

df["ATR14"] = tr.rolling(14).mean()

px.line(df.reset_index().dropna(), "timestamp", "ATR14")

### Momentum

Momentum beskriver hastigheten på prisbevegelser. Positivt momentum indikerer ofte styrke i markedet.:

$$
Momentum=
Price_t
-
Price_{t-n}
$$

F.eks:  
Kjøp når momentum > 0,  
Selg når momentum < 0

In [9]:
df["momentum"] = df["close"] - df["close"].shift(126)

px.line(df.reset_index().dropna(), "timestamp", ["momentum"]).show()
px.line(df.reset_index().dropna(), "timestamp", ["close"]).show()

### Mean Reversion

En teori som antar at priser har en tendens til å bevege seg tilbake mot sitt historiske gjennomsnitt.

Mean Reversion brukes ofte sammen med:

- Bollinger Bands
- RSI
- Z-Score

F.eks:  
Kjøp når Deviation < 0,  
Selg når Deviation > 0,  
*Anbefales å bruke et til filter, som z-score.*

In [10]:
df["deviation"] = df["close"] - df["sma50"]

px.line(df.dropna().reset_index(), "timestamp", "deviation")

### Z-Score

En indikator som måler hvor langt en observasjon befinner seg fra gjennomsnittet målt i standardavvik:

$$
Z=
\frac{
X-\mu
}
{
\sigma
}
$$

*Brukes mest som et filter.*  
F.eks:  
Kjøp når Z-Score < -2,  
Selg når Z-Score > 0

In [11]:
mean = df["close"].rolling(20).mean()
std = df["close"].rolling(20).std()

df["zscore"] = (df["close"] - mean) / std

px.line(df.dropna().reset_index(), "timestamp", "zscore")

## Target Engineering

Maskinlæring er en metode der datamaskiner lærer mønstre fra historiske data for å gjøre prediksjoner eller ta beslutninger. For eksempel kan man bruke tidligere aksjedata til å forsøke å forutsi fremtidige kursbevegelser.  


Target engineering er prosessen med å definere og konstruere målvariabelen (target) som maskinlærings modellen skal lære å predikere. For eksempel kan man i aksjeforecasting velge om målet skal være den eksakte aksjekursen om én uke (regresjon) eller om kursen går opp eller ned (klassifikasjon).

### Klassifikasjon: 
En metode i maskinlæring som brukes til å plassere data i forhåndsdefinerte kategorier eller klasser. For eksempel å avgjøre om aksjen stiger eller synker imorgen. (1 eller 0). Vi har ikke en feature (en kolonne) i vår dataframe for dette enda.

In [12]:
df["target"] = (df["close"].shift(-1) > df["close"]).astype(int).dropna() 
# Bruk dropna() fordi det vil alltid mangle en rad siden den første raden ikke har en rad før seg selv å sammenlikne med.

df["target"]

symbol  timestamp                
AAPL    2025-01-02 05:00:00+00:00    0
        2025-01-03 05:00:00+00:00    1
        2025-01-06 05:00:00+00:00    0
        2025-01-07 05:00:00+00:00    1
        2025-01-08 05:00:00+00:00    0
                                    ..
        2026-06-11 04:00:00+00:00    0
        2026-06-12 04:00:00+00:00    1
        2026-06-15 04:00:00+00:00    1
        2026-06-16 04:00:00+00:00    0
        2026-06-17 04:00:00+00:00    0
Name: target, Length: 365, dtype: int64

### Regresjon: 
En metode i maskinlæring som brukes til å forutsi en kontinuerlig numerisk verdi. For eksempel å anslå close kolonnen på en AAPL basert på ulike egenskaper eller features. I vår dataframe kan vi bruke "close" kolonnen som vår target.

## Korrelasjonsmatrise part 2

In [13]:
# Heatmap korrelasjonsmatrise for å se hvilke variabler beskriver hverandre
df = df.dropna()
corr = df.corr()

px.imshow(
    corr,
    zmin= -1,
    zmax= 1,
    text_auto= ".2f",
    color_continuous_scale = "RdBu_r"

).show()

**Som oftest ønsker man å samle de radene som er ukorrolerte med hverandre siden de viser spredningen av dataen best.**  
Som oftest jobber man mye med dataen for å få best resultat på predikasjonene når man trener en maskinlærings modell.

## Feature Selection
En enkel måte å velge de kolonnene med mest varians.

In [14]:
varians =  df.var(numeric_only = True).drop(columns = ["target"])
top_features = varians.nlargest(3).index.tolist()
df_selected = df[top_features + ["target"]]

corr = df_selected.corr()

px.imshow(
    corr,
    zmin= -1,
    zmax= 1,
    text_auto= ".2f",
    color_continuous_scale = "RdBu_r"

).show()

## PCA (Principal Component Analysis)

Du kan også bruke En metode for å redusere antall variabler ved å kombinere dem til noen få nye variabler som bevarer mest mulig av  variansen i dataene. Dette kaller vi dimensjons reduksjon.
 
Eksempel: Hvis du har 20 tekniske indikatorer for aksjer, kan PCA redusere disse til for eksempel 3–5 features som fanger opp de viktigste mønstrene i dataene.

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df.drop(columns = ["target"])
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components = 0.95) # behold komponenter som forklarer 95% av variansen

X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(X_pca.shape[1])]
)

df_pca["target"] = df["target"].values

# Husk å ta med targets også.
df_pca["target"] = df["target"].values

In [16]:
corr = df_pca.corr()

px.imshow(
    corr,
    zmin= -1,
    zmax= 1,
    text_auto= ".2f",
    color_continuous_scale = "RdBu_r"

).show()

In [17]:
px.scatter(df_pca, x = "PC1", y = "PC2").show()
px.scatter(df_pca, x = "PC2", y = "PC3").show()
px.scatter(df_pca, x = "PC3", y = "PC4").show()
px.scatter(df_pca, x = "PC4", y = "PC5").show()
px.scatter(df_pca, x = "PC1", y = "PC5").show()

## Lagring av data til fil

In [19]:
df.to_csv("df.csv", index=False)